In [ ]:
import pandas as pd
from googletrans import Translator
import time
import re
from tqdm import tqdm
from pathlib import Path

# 1. 데이터 로드
file_path = '/data/ephemeral/home/project/nlp-team2/dialogue-summarization/data/processed/train_en.csv'
output_file = '/data/ephemeral/home/project/nlp-team2/dialogue-summarization/data/processed/train_backtrans.csv'
SAVE_EVERY = 200   # N행마다 중간 저장

df = pd.read_csv(file_path)

# 이전 중간 저장본이 있으면 이어서 시작
if Path(output_file).exists():
    saved = pd.read_csv(output_file)
    start_idx = len(saved)
    results = saved['dialogue'].tolist()
    print(f"이어서 시작: {start_idx}행부터 ({len(df) - start_idx}개 남음)")
else:
    start_idx = 0
    results = []
    print(f"처음부터 시작: {len(df)}개")

def translate_with_retry(text, retries=5, delay=2.0):
    """timeout 시 Translator 재초기화 + 지수 백오프 retry."""
    if pd.isna(text) or str(text).strip() == "":
        return ""

    # #PersonN#: 접두사 분리 (번역 대상에서 제외)
    m = re.match(r'^(#Person\d+#:\s*)(.*)', str(text), re.DOTALL)
    prefix, content = (m.group(1), m.group(2)) if m else ("", str(text))

    if not content.strip():
        return text

    for attempt in range(retries):
        try:
            translator = Translator()   # 매번 새 세션으로 timeout 방지
            translated = translator.translate(content, src='en', dest='ko').text
            time.sleep(0.05)            # 가벼운 딜레이로 rate limit 방지
            return prefix + translated
        except Exception as e:
            wait = delay * (2 ** attempt)
            print(f"\n  [retry {attempt+1}/{retries}] {type(e).__name__} → {wait:.1f}초 대기")
            time.sleep(wait)

    print(f"\n  [SKIP] {retries}회 실패, 원문 유지: {text[:60]}")
    return text  # 모두 실패 시 원문 유지

# 2. 번역 실행 (이어쓰기 지원)
print("번역 시작...")
for i, row in enumerate(tqdm(df.itertuples(), total=len(df), desc="En→Ko")):
    if i < start_idx:
        continue

    translated_lines = []
    for line in str(row.dialogue).split("\n"):
        translated_lines.append(translate_with_retry(line))
    results.append("\n".join(translated_lines))

    # N행마다 중간 저장
    if (i + 1) % SAVE_EVERY == 0:
        tmp_df = df.iloc[:len(results)].copy()
        tmp_df['dialogue'] = results
        tmp_df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"\n  중간 저장: {len(results)}/{len(df)}행")

# 3. 최종 저장
result_df = df.copy()
result_df['dialogue'] = results
result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n번역 완료! 저장: {output_file} ({len(result_df)}행)")

처음부터 시작: 12457개
번역 시작...


En→Ko:   2%|▏         | 200/12457 [46:06<44:47:36, 13.16s/it]


  중간 저장: 200/12457행

  [retry 1/5] ReadTimeout → 2.0초 대기


En→Ko:   2%|▏         | 214/12457 [50:10<51:54:31, 15.26s/it]


  [retry 1/5] TypeError → 2.0초 대기


...

En→Ko: 100%|█████████▉| 12448/12457 [48:08:08<02:59, 19.91s/it]


  [retry 1/5] ReadTimeout → 2.0초 대기


En→Ko: 100%|██████████| 12457/12457 [48:10:22<00:00, 13.92s/it]


번역 완료! 저장: /data/ephemeral/home/project/nlp-team2/dialogue-summarization/data/processed/train_backtrans.csv (12457행)


: 